In [74]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_chroma import Chroma

In [75]:
from dotenv import load_dotenv



import os

load_dotenv()

print("Did it find the NVIDIA key?:", "NVIDIA_API_KEY" in os.environ)

Did it find the NVIDIA key?: True


In [76]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [77]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [78]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [79]:
books["tagged_description"].to_csv("tagged_description.txt",
index=False,
header=False)

In [80]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
documents = text_splitter.split_documents(raw_documents)

In [81]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [ ]:
my_key = "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" # use your own api-key here.

embedder = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5",
    nvidia_api_key=my_key
    )


db_books = Chroma.from_documents(
    documents,
    embedding=embedder
)

In [83]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='f165f7ec-56d9-4282-8614-dfdd23ec1186', metadata={'source': 'tagged_description.txt'}, page_content='"9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience."\n"9780786808373 Introducing your baby to birds, cats, dogs, and babies through fine art, illsutration and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child\'s drawings and abstract art to playful photos. A brief text accompanies each image, introducing baby to some basic -- and sometimes playful -- information on the subjects."'),
 Document(id='70ff6e8d-9954-4ec9-aefa-8be76d32abea', metadata={'source': 'tagged_description.txt'}, page_content='"9780374422080 This Newbery Honor Book tells the story of 11 -year-old Primrose

In [87]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip('" '))]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [92]:
import re

def retrieve_semantic_recommendations(
    query: str,
    top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=top_k*2)

    books_list = []

    for doc in recs:
        match = re.search(r'\b\d{13}\b', doc.page_content)
        if match:
            books_list.append(int(match.group()))
            
    return books[books["isbn13"].isin(books_list)].head(top_k)

In [93]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
399,9780062516374,006251637X,Rainforest Home Remedies,Rosita Arvigo;Nadine Epstein,Health & Fitness,http://books.google.com/books/content?id=DLnwO...,Rainforest Healing from Your Home and Garden F...,2001.0,4.12,221.0,65.0,Rainforest Home Remedies: The Maya Way To Heal...,9780062516374 Rainforest Healing from Your Hom...
413,9780064405850,0064405850,Strawberry Girl 60th Anniversary Edition,Lois Lenski,Juvenile Fiction,http://books.google.com/books/content?id=AQXM2...,"The land was theirs, but so were its hardships...",1995.0,3.86,208.0,10655.0,Strawberry Girl 60th Anniversary Edition,"9780064405850 The land was theirs, but so were..."
1639,9780374422080,0374422087,Everything on a Waffle,Polly Horvath,Juvenile Fiction,http://books.google.com/books/content?id=NimVJ...,This Newbery Honor Book tells the story of 11 ...,2004.0,3.71,150.0,9631.0,Everything on a Waffle,9780374422080 This Newbery Honor Book tells th...
1741,9780375814686,037581468X,Terrier,Tamora Pierce,Juvenile Fiction,http://books.google.com/books/content?id=CGScK...,"When sixteen-year-old Beka becomes ""Puppy"" to ...",2006.0,4.16,584.0,54179.0,Terrier,9780375814686 When sixteen-year-old Beka becom...
1905,9780393311037,0393311031,Hen's Teeth and Horse's Toes,Stephen Jay Gould,Nature,http://books.google.com/books/content?id=EPh9j...,A collection of essays answers questions about...,1994.0,4.10,416.0,1680.0,Hen's Teeth and Horse's Toes,9780393311037 A collection of essays answers q...
2261,9780446518628,044651862X,The Celestine Prophecy,James Redfield,Fiction,http://books.google.com/books/content?id=UXolx...,You have never read a book like this before --...,1994.0,3.63,247.0,1280.0,The Celestine Prophecy: An Adventure,9780446518628 You have never read a book like ...
2858,9780590032490,0590032496,The witches,Roald Dahl,Juvenile Nonfiction,http://books.google.com/books/content?id=tpQxo...,"A young boy and his Norwegian grandmother, who...",1997.0,4.17,208.0,254867.0,The witches,9780590032490 A young boy and his Norwegian gr...
3201,9780689823824,0689823827,A Child's Garden of Verses,Robert Louis Stevenson,Juvenile Fiction,http://books.google.com/books/content?id=luTZA...,"Here is a delightful look at childhood, writte...",1999.0,4.30,67.0,21780.0,A Child's Garden of Verses,9780689823824 Here is a delightful look at chi...
3207,9780689846175,0689846177,"Sagwa, The Chinese Siamese Cat",Amy Tan,Juvenile Fiction,http://books.google.com/books/content?id=HBe-W...,Ming Miao tells her kittens about the antics o...,2001.0,4.02,40.0,967.0,"Sagwa, The Chinese Siamese Cat",9780689846175 Ming Miao tells her kittens abou...
3211,9780689851902,0689851901,"What a Scare, Jesse Bear",Nancy White Carlstrom,Juvenile Fiction,http://books.google.com/books/content?id=VGgCA...,Jesse Bear has a wonderful time getting ready ...,2012.0,3.47,32.0,100.0,"What a Scare, Jesse Bear",9780689851902 Jesse Bear has a wonderful time ...
